# 2026/8/5
### 模型参数占用显存估计

基本单位：

**8bit = 1 Byte**

1KB = $2^{10}$ Byte

1MB = $2^{20}$ Byte

1GB = $2^{30}$ Byte

- **FP32(32-bit Floating Point)**： 每个参数占用 32 bit = 4 Byte
- **FP16 / BF16**：每个参数占用 16 bit = 2 Byte
- **INT8**：每个参数占用 8 bit = 1 Byte

    1 Billion = $10^{9}$ 个参数

    | 参数量 | FP32显存 | FP16显存 | INT8显存 |
    | --- | ------ | ------ | ------ |
    | 1B  | 约4GB   | 约2GB   | 约1GB   |
    | 7B  | 约28GB  | 约14GB  | 约7GB  |
    | 10B | 约40GB  | 约20GB  | 约10GB  |
    | 70B | 约280GB | 约140GB | 约70GB |

    <br>


    

    


# Llama架构分析

## 架构改动（在Transformer基础上）

### 1. SwiGLU(Swish-Gated Linear Unit)激活函数

  - 在Transformer的FFN层中，使用的是ReLu激活函数，其图像为

    <div align="center">
      <img src="note_pics/ReLU.png" style="width:30%">
      <br>
    </div>

    - 在FFN中的应用表达式：
      $$
      FFN(x)=W_2ReLU(W_1x+b_1)+b_2
      $$

  - Llama作者在实验中通过**炼丹**发现SwiGLU效果更优，其公式如下：

      $$
      SwiGLU(x)=W_{down}(SiLU(W_{gate}x)\odot W_{up}x)
      $$

    这里面的线性变换**都没有偏置项**
    
    输入张量维度变化：4096 -> 11008 -> 4096

    其组成有两个分支
    - 门控分支：对输入进行投影后输入Sigmoid函数，产生一个控制信号，用来决定需要保留的特征。

      $$
      gate = SiLU(W_{gate}x)
      $$
      $$
      SiLU(x)=xSigmoid(x)=\frac{x}{1+e^{-x}}
      $$

    <div align="center">
      <img src="note_pics/Sigmoid.png" style="width:30%">
      <img src="note_pics/Swish.png" style="width:30%">
      <br>
    </div>

    - 特征分支：对输入进行投影，在此作为门控的输入特征，用来进行特征提取。

      $$
      up = W_{up}x
      $$
    
    - 两个分支相乘后再进行投影

      $$
      hidden = gate \odot up
      $$  
      $$
      output=W_{down}(hidden)
      $$






    

手撕SiLU函数

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def silu(x):
    """
    SiLU(x)=x*sigmoid(x)
    """
    sig_x = 1 / (1 + torch.exp(-x))
    return x * sig_x


# 测试
x = torch.tensor([-2.0, 0.0, 2.0])

out = silu(x)

# 这个是pytorch内置的silu函数
out1 = F.silu(x)

print(out)
print(out == out1)

tensor([-0.2384,  0.0000,  1.7616])
tensor([True, True, True])


手撕SwiGLU

In [6]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        # 门控分支
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
            
        # 特征分支
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)

        # 降维
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)


    def forward(self, x):
        # x.shape: (batchsize, seq_len, d_model)
        # 1. gate分支
        gate = self.gate_proj(x)

        # 2. up分支
        up = self.up_proj(x)
        print(up.shape)

        # 3. SiLU门控
        gate = silu(gate)
        print(gate.shape)

        # 4. 两个分支逐元素相乘
        hidden = gate * up

        # 5. down projection
        output = self.down_proj(hidden)


        return output


d_model = 8
d_ff = 32

swiglu = SwiGLU(d_model, d_ff)
x = torch.randn(2,5,d_model)

out = swiglu(x)
print("输入:")
print(x.shape)

print("输出:")
print(out.shape)


torch.Size([2, 5, 32])
torch.Size([2, 5, 32])
输入:
torch.Size([2, 5, 8])
输出:
torch.Size([2, 5, 8])


###  2. RMSNorm(Root Mean Square Normalization)

- Llama 使用RMSNorm（均方根正则化）替换原有的LayerNorm

  > LayerNorm内容详见：```llm_study\week2\notes\Transformer&MoE笔记.html``` 2026/7/31笔记

- 公式
  - 计算均方根：
    $$
    RMS(x)=\sqrt{\frac{1}{n}\sum_{i=1}^{n}x_i^2+\epsilon}
    $$

  - 归一化：
    $$
    RMSNorm(x)=\gamma\frac{x}{RMS(x)}
    $$

    其中， $\gamma$ 是可学习的缩放参数，这里与LayerNorm不同，没有偏移参数 $\beta$。
    
    而且值得注意的是，与LayerNorm类似的，这里的 $\gamma$ 数量与输入向量的维度数量一致，所有token都要经过同样的一组 $\gamma$

- LLaMA 使用 RMSNorm 就是因为它更简单、计算更快，同时保留了足够的表达能力。
  

手撕RMS均方根计算

In [34]:
import torch

def rms(x, eps=1e-8):
    """
    计算 RMS

    x:
    (..., hidden_dim)

    返回:
    (..., 1)
    """
    # 保持平方和项维度不消失，便于后面操作
    x_square = torch.sum(x**2, dim=-1, keepdim=True)
    print(x_square.shape)

    x_rms = torch.sqrt(x_square / x.shape[-1] + eps)

    # 简写
    # x_rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True))
    return x_rms

x = torch.tensor(
    [
        [1.,2.,3.],
        [4.,5.,6.]
    ]
)

x = torch.randn(2,3,4)

print(x.shape)
rms_value = rms(x)

print(rms_value.shape)



torch.Size([2, 3, 4])
torch.Size([2, 3, 1])
torch.Size([2, 3, 1])


手撕RMSNorm

> RMSNorm/LayerNorm中的可学习参数 $\gamma$ 是一个长度等于 hidden_dim 的缩放向量。
> 
> 归一化后的每个 token 向量都会与**同一组 $\gamma$ 进行逐元素相乘**，即**每个 hidden 维度**都有独立的缩放参数，并且所有 token **共享**这组参数。

In [25]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-8):
        super().__init__()

        self.eps = eps

        # 可学习参数 gamma
        self.weight = nn.Parameter(
            torch.ones(dim)
        )


    def forward(self, x):
        # x:
        # (batch, seq_len, hidden_dim)
        print(x.shape)

        # 1. 计算均方根
        rms_x = rms(x, self.eps)
        print(rms_x.shape)
        
        # 2. 归一化
        output = x / rms_x * self.weight

        return output

rmsnorm = RMSNorm(16)

x = torch.randn(2, 5, 16)

x_rmsnorm = rmsnorm(x)
print(x_rmsnorm.shape)


torch.Size([2, 5, 16])
torch.Size([2, 5, 1])
torch.Size([2, 5, 16])


<div style="page-break-after: always;"></div>

# 2026/8/6
###  3. RoPE(Rotary Position Embedding)

- Llama架构使用旋转位置编码替换绝对位置编码

>APE与RoPE内容详见：```llm_study\week2\notes\Transformer&MoE笔记.html``` 2026/8/1~3笔记

  - 为什么使用绝对位置编码的模型上下文能力不足

    Transformer 中绝对位置编码：

    $$
    PE(pos,2i)=\sin(\frac{pos}{10000^{2i/d}})
    $$
    $$
    PE(pos,2i+1)=\cos(\frac{pos}{10000^{2i/d}})
    $$

    由于三角函数具有周期性，所以如果：
    
    $$
    pos_1-pos_2=2\pi\times10000^{2i/d}
    $$
    
    则
    
    $$PE(pos_1)=PE(pos_2)$$
    
    即不同位置可以产生**相似的位置编码**结果，如：

    $$
    PE(100) \approx PE(10000), PE(500)\approx PE(20000)
    $$

    所以，在token位置差距足够大时，不同位置可能映射到相似的位置编码向量
    
    **模型可能无法有效区分两个 token 的绝对位置信息**

  - 相对位置编码提升了上下文能力

    >RoPE不直接添加位置向量，而是利用注意力计算本身的特点，根据token所在位置对Query和Key进
行旋转，使Attention计算结果融入了两个token之间的相对距离（角度差），因此具有更好的长上下文外推能力。

    旋转矩阵满足
    $$
    R(i)^TR(j)=R(j-i)
    $$

  - 举例说明两者区别：
    - 绝对位置编码：

      训练时，“狗”对应的 token 为第100个位置，“猫”对应的 token 在第200个位置，模型学会了下面  的位置关系：
  
      $$
      PE(100),PE(200)
      $$
  
      推理时，“狗”对应的 token 为第10000个位置，“猫”对应的 token 在第10100个位置，而模型**没见过**这两个token之间下面这样的位置关系：
  
      $$
      PE(10000),PE(10100)
      $$
    
    - 旋转位置编码：

      训练时，“狗”对应的 token 为第100个位置，“猫”对应的 token 在第200个位置，模型学会了下面  的位置关系：
  
      $$
      200-100=100 \rightarrow R(100)
      $$
  
      推理时，“狗”对应的 token 为第10000个位置，“猫”对应的 token 在第10100个位置，而模型**已经见过**这两个token之间下如此的**相对位置**关系：
  
      $$
      10100-10000=100 \rightarrow R(100)
      $$
    
      



### 4. GQA(Group Query Attention)

**注：LLaMA 1 主要是 MHA，LLaMA 2 的 70B 和后来的 LLaMA 3 系列明确使用 GQA**
<div align="center">

<img src="note_pics/GQA.png" width="60%">

</div>

- MHA(Multi Head Attention)
  
  Transformer中的多头注意力机制在计算的时候将**一个token分成了多个等维度的向量**，在计算时使用多个并列的头来分别进行注意力计算，从而能够在保证特征提取效果的基础上**实现并行计算**。

  注意力计算包括当前 token 以及之前的所有 token 的 KV ，所以这里会使用**KV Cache**来加速运算。但是多头注意力机制对于每一个 Query head 都有各自的一组 Key 和 Value head，当模型体量较大时会**占用大量显存而且计算量很大**。如Llama模型配置如下：

  $$
  layers=32,
  heads=32,
  head_{dim}=128,
  context=4096,
  fp16
  $$
  
  KV Cache:
  
  $$
  2 \times layers \times seq \times heads \times dim
  $$
  
  第一项的 2 代表 K 和 V 这两次计算，带入数值可得一次注意力计算所需要**缓存的元素数量**以及**占用显存**：
  
  $$
  2×32×4096×32×128 \approx 2GB 
  $$

- MQA(Multi Query Attention)

  由Google提出，核心思想是所有 Query Head 共享一组 K、V，从而极大减少显存占用。

  所有 Query heads 共享同一组由 KV 投影生成的 Key 和 Value，从而进行注意力计算，KV Cache 占用减少到原来的 32 分之一：
  
  $$
  Q:
  (batch, 32, seq, 128)
  $$
  $$
  K:
  (batch, 1, seq, 128)
  $$
  $$
  V:
  (batch, 1, seq, 128)
  $$

  但是，这种方式减少了 K/V head 的数量，因此降低了不同 Query head 可访问的 K/V 表征多样性（一个 token 从不同角度被理解的机会减少），可能损失表达能力。

- GQA(Group Query Attention)

  改变了 MQA 这样“极端”的方式：不让所有 Query 共享，而是几个 Query 共享一组 KV。比如 $head=32$ :
  
  $$
  Q_{1}...Q_{4} \rightarrow K_{1} V_{1}
  $$
  $$
  Q_{5}...Q_{8} \rightarrow K_{2} V_{2}
  $$ 
  $$
  ...
  $$
  $$
  Q_{29}...Q_{32} \rightarrow K_{8} V_{8}
  $$
  
  > 这里一开始有一个理解误区：输入token不是映射成 ```num_heads * head_dim``` 后再根据组数挑其中几个 head。 token 的 hidden state 仍然是完整的 hidden_dim，K/V 投影层直接把它映射成 ```group_num * head_dim```。
  
  这样 KV 就只需要生成并缓存 8 个 heads，在保留表征能力的同时减少显存占用。




In [4]:
import torch
import torch.nn as nn
import math

# ==========================================
# 1. RoPE (Rotary Position Embedding) 实现
# ==========================================
class RoPE(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        
        # 生成频率序列 theta (对应笔记中的 1/10000^(2i/d))
        # theta_i = base^(-2i/d), i = 0, 1, ..., d/2-1
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, seq_len=None):
        """
        x: [batch_size, seq_len, n_heads, head_dim]
        """
        if seq_len is None:
            seq_len = x.shape[1]
            
        # 生成位置索引 pos: [0, 1, ..., seq_len-1]
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        
        # 计算旋转角度 m * theta
        # freqs: [seq_len, head_dim/2]
        freqs = torch.outer(t, self.inv_freq) 
        
        # 构建旋转矩阵所需的 sin 和 cos
        # emb: [seq_len, head_dim] -> 将 [sin, cos] 拼接
        emb = torch.cat((freqs, freqs), dim=-1)
        
        # 注意：实际工程中通常直接计算 cos(m*theta) 和 sin(m*theta)
        # 然后利用复数乘法或特定的 rotate_half 函数来实现旋转
        cos = emb.cos()[None, :, None, :]
        sin = emb.sin()[None, :, None, :]
        
        return cos, sin

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    将 RoPE 应用到 q 和 k 上
    q, k: [batch, seq_len, n_heads, head_dim]
    cos, sin: [1, seq_len, head_dim] (广播机制会自动匹配 batch 和 heads 维度)
    """
    # 定义旋转操作：将向量后半部分取反并交换位置
    # 对应数学公式中的 R(x) 操作
    def rotate_half(x):
        x1 = x[..., :x.shape[-1]//2]
        x2 = x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim=-1)

    # 执行旋转公式: x * cos + rotate_half(x) * sin
    # 这里利用了广播机制，cos/sin 会自动匹配 seq_len 和 head_dim 维度
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    
    return q_embed, k_embed


# ==========================================
# 2. GQA (Grouped Query Attention) 实现
# ==========================================
class GQAAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config['n_heads']       # Query 的头数 (例如 32)
        self.n_kv_heads = config['n_kv_heads'] # KV 的头数 (例如 8)
        self.head_dim = config['head_dim']     # 每个头的维度 (例如 128)
        
        assert self.n_heads % self.n_kv_heads == 0
        self.n_rep = self.n_heads // self.n_kv_heads # 每个 KV 头被重复的次数 (例如 4)

        # 定义线性层
        # Q 有 n_heads 个输出
        self.wq = nn.Linear(config['dim'], self.n_heads * self.head_dim, bias=False)
        # K, V 只有 n_kv_heads 个输出 (这是省显存的关键！)
        self.wk = nn.Linear(config['dim'], self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config['dim'], self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_heads * self.head_dim, config['dim'], bias=False)

        self.rope = RoPE(dim=self.head_dim)

    def forward(self, x):
        bsz, seq_len, _ = x.shape
        
        # 1. 投影生成 Q, K, V
        xq = self.wq(x).view(bsz, seq_len, self.n_heads, self.head_dim)
        xk = self.wk(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim)
        xv = self.wv(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim)

        # 2. 应用 RoPE (获取 cos/sin 并应用)
        cos, sin = self.rope(xq)
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)

        # 3. GQA 核心逻辑：KV 头复用 (Repeating)
        # 将 K 和 V 从 [bsz, seq, n_kv_heads, dim] 
        # 扩展为 [bsz, seq, n_heads, dim] 以便与 Q 进行矩阵乘法
        # 这里的逻辑是：K1, K2 -> K1, K1, K1, K1, K2, K2, K2, K2
        xk = xk.repeat_interleave(self.n_rep, dim=2)
        xv = xv.repeat_interleave(self.n_rep, dim=2)

        # 4. 计算注意力分数 (简化版，未做 Mask 和 Softmax 优化)
        # xq/xk/xv: [bsz, seq, heads, dim] -> [bsz, heads, seq, dim]
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)

        scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = torch.softmax(scores.float(), dim=-1).type_as(xq)
        
        output = torch.matmul(scores, xv)
        output = output.transpose(1, 2).contiguous().view(bsz, seq_len, -1)
        
        return self.wo(output)

# ==========================================
# 测试代码
# ==========================================
if __name__ == "__main__":
    # 模拟 LLaMA 配置
    config = {
        'dim': 4096,
        'n_heads': 32,      # Query Heads
        'n_kv_heads': 8,    # KV Heads (GQA 的关键，如果是 MHA 则这里是 32)
        'head_dim': 128
    }

    model = GQAAttention(config)
    
    # 输入张量: Batch=1, SeqLen=10, Dim=4096
    dummy_input = torch.randn(1, 10, 4096)
    
    print(f"输入形状: {dummy_input.shape}")
    
    # 前向传播
    output = model(dummy_input)
    
    print(f"输出形状: {output.shape}")
    print(f"模型参数量统计:")
    print(f"  WQ 参数: {config['dim']} * {config['n_heads'] * config['head_dim']}")
    print(f"  WK 参数: {config['dim']} * {config['n_kv_heads'] * config['head_dim']} (显著小于 WQ)")

输入形状: torch.Size([1, 10, 4096])
输出形状: torch.Size([1, 10, 4096])
模型参数量统计:
  WQ 参数: 4096 * 4096
  WK 参数: 4096 * 1024 (显著小于 WQ)


# 2026/8/12

## Llama3.1-8B 模型分析

### 1. 配置文件


In [1]:
import json

with open("../Meta-Llama-3-8B/config.json", "r") as f:
    config = json.load(f)

config

{'architectures': ['LlamaForCausalLM'],
 'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 128000,
 'eos_token_id': 128001,
 'hidden_act': 'silu',
 'hidden_size': 4096,
 'initializer_range': 0.02,
 'intermediate_size': 14336,
 'max_position_embeddings': 8192,
 'model_type': 'llama',
 'num_attention_heads': 32,
 'num_hidden_layers': 32,
 'num_key_value_heads': 8,
 'pretraining_tp': 1,
 'rms_norm_eps': 1e-05,
 'rope_scaling': None,
 'rope_theta': 500000.0,
 'tie_word_embeddings': False,
 'torch_dtype': 'bfloat16',
 'transformers_version': '4.40.0.dev0',
 'use_cache': True,
 'vocab_size': 128256}

个别配置参数功能解释

- intermediate_size：用在 LLaMA 的 SwiGLU MLP 里

  $$
  gate_{proj}: 4096 \rightarrow 14336
  $$
  $$
  up_{proj}:   4096 \rightarrow 14336
  $$
  $$
  down_{proj}: 14336 \rightarrow 4096
  $$
  
  用于设置FFN层中的维度扩张之后的大小

下面的参数是 Llama3.1 用到的配置，主要是改动旋转位置编码从而提升位置编码范围

- max_position_embeddings：

  模型配置上支持的最大序列长度，即最多约 131072 个 token，也就是 128K 上下文。它会影响位置编码的可用范围，以及推理时 KV Cache 可能达到的长度。

  但这不代表模型只要设成 131072 就天然擅长 128K。模型是否真的能用好长文本，还取决于训练数据、长上下文训练，以及 RoPE 的缩放方式。

- rope_scaling：

  这是为了让 RoPE 更好地扩展到长上下文的配置。这份模型原始 RoPE 长度是 8192，但目标长度是 131072，即 8192 × 16

  这里采用的是 LLaMA 3 的 rope_type: "llama3" 方案：它不会简单地把全部频率统一缩放，而是根据频率区别处理：

    low_freq_factor: 1.0：低频部分基本保持，保留较稳定的长距离位置信息（因为长距离时主要靠token的较低维度来区分位置关系）。

    high_freq_factor: 4.0：高频部分缩放更明显，避免位置过远时旋转过快、泛化变差。

    factor: 8.0：参与这套缩放策略的总体倍率参数。

Llama架构的残差链接和 Norm 操作顺序与 Transformer 有所不同：

- 原始 Transformer(Post-LN)：先进行子层计算，再 Add，再 Norm

  梯度必须经过 LayerNorm 和多个子层。

  当 Transformer 很深时：梯度容易消失、训练初期不稳定、需要 warmup 技巧

- LLaMA(Pre-LN)：先 Norm，再进行子层计算，最后 Add

  这样存在一条直接的梯度通道，保证梯度可以直接传播。

  所以更容易训练深层 Transformer、对大模型更友好

<div align="center">

<img src="note_pics/Llama架构.jpg" width="20%">

</div>